In [1]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_excel("merged_data.xlsx")

print(df.shape)
df.head()

(367437, 24)


,restaurant_id,original_name,original_address,matched_name,google_id,matched_address,latitude,longitude,day,hour,...,weekday,demand_month,taxi_zone_id,taxi_zone_name,estimated_area_sqft,capacity,turnover_rate,takeaway_ratio,eat_in_ratio,dropoff_count
0,50142786,DA ANDREA RISTORANTE,"160 8 AVENUE, MANHATTAN, NY 10011",Da Andrea Chelsea,0x89c2593ba314c0d1:0xd5579301e987a1bd,NaN,40.742092,-74.000594,7,6,...,6,6,90,Flatiron,2143.2,117.876,1.0,0.1,0.9,31.0
1,50142786,DA ANDREA RISTORANTE,"160 8 AVENUE, MANHATTAN, NY 10011",Da Andrea Chelsea,0x89c2593ba314c0d1:0xd5579301e987a1bd,NaN,40.742092,-74.000594,7,7,...,6,6,90,Flatiron,2143.2,117.876,1.0,0.1,0.9,47.0
2,50142786,DA ANDREA RISTORANTE,"160 8 AVENUE, MANHATTAN, NY 10011",Da Andrea Chelsea,0x89c2593ba314c0d1:0xd5579301e987a1bd,NaN,40.742092,-74.000594,7,8,...,6,6,90,Flatiron,2143.2,117.876,1.0,0.1,0.9,124.0
3,50142786,DA ANDREA RISTORANTE,"160 8 AVENUE, MANHATTAN, NY 10011",Da Andrea Chelsea,0x89c2593ba314c0d1:0xd5579301e987a1bd,NaN,40.742092,-74.000594,7,9,...,6,6,90,Flatiron,2143.2,117.876,1.0,0.1,0.9,200.0
4,50142786,DA ANDREA RISTORANTE,"160 8 AVENUE, MANHATTAN, NY 10011",Da Andrea Chelsea,0x89c2593ba314c0d1:0xd5579301e987a1bd,NaN,40.742092,-74.000594,7,10,...,6,6,90,Flatiron,2143.2,117.876,1.0,0.1,0.9,249.0


In [2]:
import re
import numpy as np
import pandas as pd

def parse_typical_time(x):
    if pd.isna(x):
        return np.nan

    x = str(x).lower().strip()

    # uniform sign
    x = x.replace("–", "-").replace("—", "-")

    pattern = r"""
        (?:
            spend\s+
        )?
        (\d+(?:\.\d+)?)                      # find the first figure
        \s*
        (min|mins|minute|minutes|hr|hrs|hour|hours)?   # first unit can be empty
        \s*
        (?:
            -|to
        )?
        \s*
        (?:
            (\d+(?:\.\d+)?)                  # second fighre
            \s*
            (min|mins|minute|minutes|hr|hrs|hour|hours)? # second unit
        )?
    """

    match = re.search(pattern, x, re.VERBOSE)

    if not match:
        return np.nan

    v1 = float(match.group(1))
    u1 = match.group(2)

    v2 = match.group(3)
    u2 = match.group(4)

    def normalize_unit(unit):
        if unit in ["hr", "hrs", "hour", "hours"]:
            return "hour"
        elif unit in ["min", "mins", "minute", "minutes"]:
            return "minute"
        return None

    u1 = normalize_unit(u1)
    u2 = normalize_unit(u2)

    # if one unit missing, assume two units are the same
    if v2 is not None:
        if u2 is None and u1 is not None:
            u2 = u1
        elif u1 is None and u2 is not None:
            u1 = u2

    if u1 is None:
        return np.nan

    def to_minutes(value, unit):
        if unit == "hour":
            return value * 60
        elif unit == "minute":
            return value
        return np.nan

    t1 = to_minutes(v1, u1)

    if v2 is not None:
        t2 = to_minutes(float(v2), u2)
        return (t1 + t2) / 2

    return t1


df["typical_time_mid"] = df["typical_time_spent"].apply(parse_typical_time)


In [3]:
df[["typical_time_spent","typical_time_mid"]].head(20)

,typical_time_spent,typical_time_mid
0,People typically spend 1.5-4 hours here,165.0
1,People typically spend 1.5-4 hours here,165.0
2,People typically spend 1.5-4 hours here,165.0
3,People typically spend 1.5-4 hours here,165.0
4,People typically spend 1.5-4 hours here,165.0
5,People typically spend 1.5-4 hours here,165.0
6,People typically spend 1.5-4 hours here,165.0
7,People typically spend 1.5-4 hours here,165.0
8,People typically spend 1.5-4 hours here,165.0
9,People typically spend 1.5-4 hours here,165.0


In [4]:
df["popularity"] = (
    df["rating"]
    * np.log1p(df["reviews"])
)

In [5]:
df["hour_sin"] = np.sin(
    2*np.pi*df["hour"]/24
)

df["hour_cos"] = np.cos(
    2*np.pi*df["hour"]/24
)

In [6]:
df["friday"] = (df["day"]==5).astype(int)

df["saturday"] = (df["day"]==6).astype(int)

df["sunday"] = (df["day"]==7).astype(int)

In [7]:
df["ln_area"] = np.log(df["estimated_area_sqft"])

In [8]:
df["ln_dropoff"] = np.log1p(df["dropoff_count"])

In [9]:
df["ln_reviews"] = np.log1p(df["reviews"])

In [10]:
restaurant_ids = df["restaurant_id"].unique()

print(len(restaurant_ids))

2815


In [11]:
train_restaurants = restaurant_ids[:155]

test_restaurants = restaurant_ids[155:]

In [12]:
train_df = df[df["restaurant_id"].isin(train_restaurants)].copy()

test_df = df[df["restaurant_id"].isin(test_restaurants)].copy()

print(train_df.shape)
print(test_df.shape)

(20529, 34)
(346908, 34)


In [13]:
feature_cols = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
]

target_col = "busyness_score"

train_model_df = train_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

test_model_df = test_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

In [14]:
X_train = train_model_df[feature_cols]
X_train = sm.add_constant(X_train)

y_train = train_model_df[target_col]

X_test = test_model_df[feature_cols]
X_test = sm.add_constant(X_test)

y_test = test_model_df[target_col]

model = sm.OLS(y_train, X_train)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:         busyness_score   R-squared:                       0.299
Model:                            OLS   Adj. R-squared:                  0.298
Method:                 Least Squares   F-statistic:                     673.5
Date:                Thu, 09 Jul 2026   Prob (F-statistic):               0.00
Time:                        21:26:06   Log-Likelihood:                -58944.
No. Observations:               12656   AIC:                         1.179e+05
Df Residuals:                   12647   BIC:                         1.180e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               42.8951      2.964  

In [15]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_pred = results.predict(X_train)
test_pred = results.predict(X_test)

print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.2987489363488868
MAE = 20.672777234902757
RMSE = 25.49461703070775

Test
R2 = 0.2985221260182789
MAE = 20.340670293928426
RMSE = 25.359633931248634


In [16]:
feature_cols = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
    "ln_area",
    "takeaway_ratio"
]

target_col = "busyness_score"

train_model_df = train_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

test_model_df = test_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

In [17]:
X_train = train_model_df[feature_cols]
X_train = sm.add_constant(X_train)

y_train = train_model_df[target_col]

X_test = test_model_df[feature_cols]
X_test = sm.add_constant(X_test)

y_test = test_model_df[target_col]

model = sm.OLS(y_train, X_train)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:         busyness_score   R-squared:                       0.299
Model:                            OLS   Adj. R-squared:                  0.299
Method:                 Least Squares   F-statistic:                     540.6
Date:                Thu, 09 Jul 2026   Prob (F-statistic):               0.00
Time:                        21:26:06   Log-Likelihood:                -58937.
No. Observations:               12656   AIC:                         1.179e+05
Df Residuals:                   12645   BIC:                         1.180e+05
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               35.5277      3.586  

In [18]:
train_pred = results.predict(X_train)
test_pred = results.predict(X_test)

print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.2994990253200348
MAE = 20.65023930125811
RMSE = 25.480978300908355

Test
R2 = 0.2999795756282141
MAE = 20.298836809296002
RMSE = 25.333275575605654


In [19]:
feature_cols = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
    "ln_dropoff",
    "ln_area",
    "takeaway_ratio"  
]


target_col = "busyness_score"

train_model_df = train_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

test_model_df = test_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

In [20]:
X_train = train_model_df[feature_cols]
X_train = sm.add_constant(X_train)

y_train = train_model_df[target_col]

X_test = test_model_df[feature_cols]
X_test = sm.add_constant(X_test)

y_test = test_model_df[target_col]

model = sm.OLS(y_train, X_train)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:         busyness_score   R-squared:                       0.300
Model:                            OLS   Adj. R-squared:                  0.300
Method:                 Least Squares   F-statistic:                     493.4
Date:                Thu, 09 Jul 2026   Prob (F-statistic):               0.00
Time:                        21:26:06   Log-Likelihood:                -58930.
No. Observations:               12656   AIC:                         1.179e+05
Df Residuals:                   12644   BIC:                         1.180e+05
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               38.6578      3.674  

In [21]:
train_pred = results.predict(X_train)
test_pred = results.predict(X_test)

print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.3003308488343247
MAE = 20.618076553824046
RMSE = 25.465844865176173

Test
R2 = 0.2963471178995336
MAE = 20.34674107821917
RMSE = 25.398879792584157


In [22]:
features = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
    "ln_dropoff",
    "ln_area",
    "takeaway_ratio"
]

In [23]:
X_train = train_model_df[features]
y_train = train_model_df["busyness_score"]

X_test = test_model_df[features]
y_test = test_model_df["busyness_score"]

In [24]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

train_pred = rf.predict(X_train)
test_pred = rf.predict(X_test)

In [25]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.8681881534705045
MAE = 7.631370684707964
RMSE = 11.053223408007112

Test
R2 = 0.33402837145336894
MAE = 18.22960505022334
RMSE = 24.70945640184873


In [26]:
importance = (
    pd.Series(
        rf.feature_importances_,
        index=features
    )
    .sort_values(ascending=False)
)

print(importance)

hour_sin            0.321048
hour_cos            0.132475
takeaway_ratio      0.108563
typical_time_mid    0.102663
ln_area             0.083159
ln_reviews          0.079309
rating              0.076871
ln_dropoff          0.062482
saturday            0.016742
friday              0.011387
sunday              0.005301
dtype: float64


In [27]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [28]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

train_pred = xgb.predict(X_train)
test_pred = xgb.predict(X_test)

In [29]:
print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.9147303104400635
MAE = 6.3234357833862305
RMSE = 8.890150865405376

Test
R2 = 0.3600533604621887
MAE = 18.24001693725586
RMSE = 24.22184456036018


In [30]:
importance = (
    pd.Series(
        xgb.feature_importances_,
        index=features
    )
    .sort_values(ascending=False)
)

print(importance)

hour_sin            0.205815
takeaway_ratio      0.163012
hour_cos            0.121607
typical_time_mid    0.099092
rating              0.093237
ln_reviews          0.083475
ln_area             0.079475
friday              0.047634
saturday            0.045180
ln_dropoff          0.037498
sunday              0.023975
dtype: float32
